In [112]:
import sys

In [113]:
sys.executable

'/Users/ellie/Documents/GitHub/creative-economy/.venv/bin/python'

In [114]:
import pandas as pd

In [115]:
df = pd.read_csv("../data/raw/wdi_data.csv")

In [116]:
df.shape

(93, 15)

In [117]:
df = df.dropna(how="all")

In [118]:
df=df.dropna(subset=["Country Code"])

In [119]:
df.shape

(88, 15)

In [120]:
df.tail()

,Country Name,Country Code,Series Name,Series Code,2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
83,Small states,SST,Merchandise imports (current US$),TM.VAL.MRCH.CD.WT,78816781584.8096,80779280258.831,88327031399.1726,100069714649.411,97955051978.36,82282160284.2898,108390792404.795,126446197093.455,129371706269.227,130992357059.51,141378070966.373
84,Small states,SST,"Trademark applications, nonresident, by count",IP.TMK.NRCT,..,..,..,..,..,..,..,..,..,..,..
85,Small states,SST,"Trademark applications, resident, by count",IP.TMK.RSCT,..,..,..,..,..,..,..,..,..,..,..
86,Small states,SST,"Industrial design applications, nonresident, b...",IP.IDS.NRCT,..,..,..,..,..,..,..,..,..,..,..
87,Small states,SST,"Industrial design applications, resident, by c...",IP.IDS.RSCT,..,..,..,..,..,..,..,..,..,..,..


Wide to long - turn dataset into one row per country × indicator × year

In [121]:
long = df.melt(
    id_vars=["Country Name", "Country Code", "Series Name", "Series Code"],
    var_name="Year",
    value_name="Value",
)

In [122]:
long.shape

(968, 6)

In [123]:
long.head()

,Country Name,Country Code,Series Name,Series Code,Year,Value
0,Madagascar,MDG,Individuals using the Internet (% of population),IT.NET.USER.ZS,2015 [YR2015],4.173999786
1,Madagascar,MDG,Logistics performance index: Overall (1=low to...,LP.LPI.OVRL.XQ,2015 [YR2015],..
2,Madagascar,MDG,Gini index,SI.POV.GINI,2015 [YR2015],..
3,Madagascar,MDG,"Charges for the use of intellectual property, ...",BX.GSR.ROYL.CD,2015 [YR2015],15610274.7936644
4,Madagascar,MDG,"Population, total",SP.POP.TOTL,2015 [YR2015],25426702


In [124]:
long["Year"] = long["Year"].str[:4].astype(int)
long["Year"].unique()

array([2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025])

In [125]:
long["Year"].unique()

array([2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025])

Clean the values and drop the empty rows

In [126]:
long["Value"] = pd.to_numeric(long["Value"], errors="coerce")
long = long.dropna(subset=["Value"])
long.shape

(620, 6)

Create dictionary to link each World Bank code to short name, CEDM element and unit

In [127]:
labels = {
    "IT.NET.USER.ZS":    ("internet_users",         "Infrastructure",      "% of population"),
    "LP.LPI.OVRL.XQ":    ("logistics_performance",  "Infrastructure",      "index, 1 to 5"),
    "SI.POV.GINI":       ("gini",                   "Governance",          "index, 0 to 100"),
    "BX.GSR.ROYL.CD":    ("ip_receipts",            "Creative value & IP", "current US$"),
    "IP.TMK.RSCT":       ("trademarks_resident",    "Creative value & IP", "applications"),
    "IP.TMK.NRCT":       ("trademarks_nonresident", "Creative value & IP", "applications"),
    "IP.IDS.RSCT":       ("designs_resident",       "Creative value & IP", "applications"),
    "IP.IDS.NRCT":       ("designs_nonresident",    "Creative value & IP", "applications"),
    "SP.POP.TOTL":       ("population",             "Helper",              "people"),
    "TX.VAL.MRCH.CD.WT": ("merch_exports",          "Helper",              "current US$"),
    "TM.VAL.MRCH.CD.WT": ("merch_imports",          "Helper",              "current US$"),
}

long["indicator"] = long["Series Code"].map(lambda code: labels[code][0])
long["element"]   = long["Series Code"].map(lambda code: labels[code][1])
long["unit"]      = long["Series Code"].map(lambda code: labels[code][2])
long["source"]    = "World Bank WDI"
long.head()

,Country Name,Country Code,Series Name,Series Code,Year,Value,indicator,element,unit,source
0,Madagascar,MDG,Individuals using the Internet (% of population),IT.NET.USER.ZS,2015,4.174000e+00,internet_users,Infrastructure,% of population,World Bank WDI
3,Madagascar,MDG,"Charges for the use of intellectual property, ...",BX.GSR.ROYL.CD,2015,1.561027e+07,ip_receipts,Creative value & IP,current US$,World Bank WDI
4,Madagascar,MDG,"Population, total",SP.POP.TOTL,2015,2.542670e+07,population,Helper,people,World Bank WDI
5,Madagascar,MDG,Merchandise exports (current US$),TX.VAL.MRCH.CD.WT,2015,2.048000e+09,merch_exports,Helper,current US$,World Bank WDI
6,Madagascar,MDG,Merchandise imports (current US$),TM.VAL.MRCH.CD.WT,2015,2.795000e+09,merch_imports,Helper,current US$,World Bank WDI


Standardize column names

In [128]:
wdi = long.rename(columns={"Country Code": "iso3", "Country Name": "country","Year": "year", "Value": "value"})
wdi = wdi[["iso3", "country", "indicator", "element", "year", "value", "unit", "source"]]
wdi.head()

,iso3,country,indicator,element,year,value,unit,source
0,MDG,Madagascar,internet_users,Infrastructure,2015,4.174000e+00,% of population,World Bank WDI
3,MDG,Madagascar,ip_receipts,Creative value & IP,2015,1.561027e+07,current US$,World Bank WDI
4,MDG,Madagascar,population,Helper,2015,2.542670e+07,people,World Bank WDI
5,MDG,Madagascar,merch_exports,Helper,2015,2.048000e+09,current US$,World Bank WDI
6,MDG,Madagascar,merch_imports,Helper,2015,2.795000e+09,current US$,World Bank WDI


In [129]:
wdi[(wdi["iso3"] == "MDG") & (wdi["indicator"] == "internet_users") & (wdi["year"] == 2024)]

,iso3,country,indicator,element,year,value,unit,source
792,MDG,Madagascar,internet_users,Infrastructure,2024,18.6779,% of population,World Bank WDI


In [130]:
df2 = pd.read_csv("../data/raw/wgi_data.csv")
df2.tail()

,Country Name,Country Code,Series Name,Series Code,2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,Data from database: Worldwide Governance Indic...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,Last Updated: 09/25/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [131]:
df2=df2.dropna(subset=["Country Code"])

In [132]:
df2.tail()

,Country Name,Country Code,Series Name,Series Code,2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
11,Comoros,COM,Government Effectiveness - Governance score (0...,GOV_WGI_GE.SC,25.511171,26.557747,27.659063,27.243534,26.728551,26.147848,22.912198,25.877874,28.214047,29.533886,28.939669
12,Seychelles,SYC,Rule of Law - Governance estimate (approx. -2....,GOV_WGI_RL.EST,0.289311,0.334444,0.280274,0.454597,0.446954,0.534170,0.553183,0.565204,0.559320,0.575887,0.577935
13,Seychelles,SYC,Rule of Law - Governance score (0-100),GOV_WGI_RL.SC,63.398742,64.142431,63.249839,66.122281,65.996342,67.433466,67.746757,67.944840,67.847891,68.120875,68.154623
14,Seychelles,SYC,Government Effectiveness - Governance estimate...,GOV_WGI_GE.EST,0.641432,0.798738,0.621827,0.642440,0.565509,0.508390,0.562876,0.424886,0.545538,0.648848,0.765515
15,Seychelles,SYC,Government Effectiveness - Governance score (0...,GOV_WGI_GE.SC,65.472846,68.692665,65.071573,65.493484,63.918819,62.749688,63.864925,61.040477,63.510051,65.624640,68.012641


In [133]:
df2=df2.rename(columns={"Country Name": "country", "Country Code": "iso3", })

In [134]:
long2 = df2.melt(
    id_vars=["country", "iso3", "Series Name", "Series Code"],
    var_name="year",
    value_name="value",
)

In [135]:
long2.tail()

,country,iso3,Series Name,Series Code,year,value
171,Comoros,COM,Government Effectiveness - Governance score (0...,GOV_WGI_GE.SC,2025 [YR2025],28.939669
172,Seychelles,SYC,Rule of Law - Governance estimate (approx. -2....,GOV_WGI_RL.EST,2025 [YR2025],0.577935
173,Seychelles,SYC,Rule of Law - Governance score (0-100),GOV_WGI_RL.SC,2025 [YR2025],68.154623
174,Seychelles,SYC,Government Effectiveness - Governance estimate...,GOV_WGI_GE.EST,2025 [YR2025],0.765515
175,Seychelles,SYC,Government Effectiveness - Governance score (0...,GOV_WGI_GE.SC,2025 [YR2025],68.012641


In [136]:
long2["year"] = long2["year"].str[:4].astype(int)
long2["year"].unique()

array([2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025])

Clean the values and drop the empty rows

In [137]:
long2["value"] = pd.to_numeric(long2["value"], errors="coerce")
long2 = long2.dropna(subset=["value"])
long2.shape

(176, 6)

In [138]:
long2.head()

,country,iso3,Series Name,Series Code,year,value
0,Madagascar,MDG,Rule of Law - Governance estimate (approx. -2....,GOV_WGI_RL.EST,2015,-0.787641
1,Madagascar,MDG,Rule of Law - Governance score (0-100),GOV_WGI_RL.SC,2015,45.652993
2,Madagascar,MDG,Government Effectiveness - Governance estimate...,GOV_WGI_GE.EST,2015,-0.971384
3,Madagascar,MDG,Government Effectiveness - Governance score (0...,GOV_WGI_GE.SC,2015,32.460945
4,Mauritius,MUS,Rule of Law - Governance estimate (approx. -2....,GOV_WGI_RL.EST,2015,0.910151


In [145]:
wgi_codes = ["GOV_WGI_RL.EST", "GOV_WGI_GE.EST"]
df2 = df2[df2["Series Code"].isin(wgi_codes)]
df2["Series Code"].unique()

<StringArray>
['GOV_WGI_RL.EST', 'GOV_WGI_GE.EST']
Length: 2, dtype: str

In [146]:
wgi_labels = {
    "GOV_WGI_RL.EST": ("rule_of_law",              "Governance", "estimate, world average = 0"),
    "GOV_WGI_GE.EST": ("government_effectiveness", "Governance", "estimate, world average = 0"),
}

long2["indicator"] = long2["Series Code"].map(lambda code: wgi_labels[code][0])
long2["element"]   = long2["Series Code"].map(lambda code: wgi_labels[code][1])
long2["unit"]      = long2["Series Code"].map(lambda code: wgi_labels[code][2])
long2["source"]    = "World Bank WGI"

KeyError: 'GOV_WGI_RL.SC'